# Chapter 2: Back-of-the-Envelope Estimation
*System Design Interview*

## TL;DR

Back-of-the-envelope estimation is a core system design skill. You need to:

1. **Know your powers of two** (data volume units)
2. **Memorize key latency numbers** (memory vs disk vs network)
3. **Understand availability nines** (99.9% = 8.77 hrs downtime/year)
4. **Estimate QPS, storage, and bandwidth** from user counts and usage patterns

> "Back-of-the-envelope calculations are estimates you create using a combination of thought experiments and common performance numbers to get a good feel for which designs will meet your requirements." -- Jeff Dean, Google

## Requirements

For any estimation problem, clearly state:
- **Assumptions:** MAU, DAU, usage frequency, data sizes
- **Units:** Always label (KB, MB, GB, QPS)
- **Rounding:** Use round numbers -- precision is NOT the goal
- **Process:** Show your reasoning step by step

## Power of Two -- Data Volume Units

In [ ]:
# Power of Two reference table
units = [
    ("1 Byte",    "8 bits",           1),
    ("1 KB",      "1,024 Bytes",      1024),
    ("1 MB",      "1,024 KB",         1024**2),
    ("1 GB",      "1,024 MB",         1024**3),
    ("1 TB",      "1,024 GB",         1024**4),
    ("1 PB",      "1,024 TB",         1024**5),
]

print(f"{'Unit':<10} {'Equivalent':<20} {'Bytes':>20}")
print("-" * 52)
for name, equiv, val in units:
    print(f"{name:<10} {equiv:<20} {val:>20,}")

print("\n--- Quick mental math helpers ---")
print(f"1 million chars (ASCII) = {1_000_000 / 1024**2:.1f} MB")
print(f"1 billion chars (ASCII) = {1_000_000_000 / 1024**3:.2f} GB")
print(f"1 high-res image (~2 MB) x 1M users = {2 * 1_000_000 / 1024**2:.0f} TB")

## Latency Numbers Every Programmer Should Know

In [ ]:
# Latency numbers (approximate, 2020 era)
latencies = [
    ("L1 cache reference",                   1,        "ns"),
    ("Branch mispredict",                     3,        "ns"),
    ("L2 cache reference",                    4,        "ns"),
    ("Mutex lock/unlock",                     17,       "ns"),
    ("Main memory reference",                100,       "ns"),
    ("Compress 1 KB (Zippy)",              2_000,       "ns"),
    ("Send 2 KB over 1 Gbps network",     11_000,       "ns"),
    ("Read 1 MB sequentially (memory)",    3_000,       "ns"),
    ("Round trip within same datacenter", 500_000,       "ns"),
    ("Disk seek",                       2_000_000,       "ns"),
    ("Read 1 MB sequentially (disk)",   825_000,        "ns"),
    ("Send packet CA -> NL -> CA",   150_000_000,       "ns"),
]

print(f"{'Operation':<45} {'Latency':>15}  {'Human Scale':>15}")
print("-" * 78)
for op, ns, unit in latencies:
    if ns < 1_000:
        human = f"{ns} ns"
    elif ns < 1_000_000:
        human = f"{ns/1_000:.1f} us"
    elif ns < 1_000_000_000:
        human = f"{ns/1_000_000:.1f} ms"
    else:
        human = f"{ns/1_000_000:.0f} ms"
    print(f"{op:<45} {ns:>12,} ns  {human:>15}")

print("\n--- Key takeaways ---")
print("1. Memory is FAST, disk is SLOW (100x-1000x difference)")
print("2. Compress before sending over the network")
print("3. Avoid disk seeks if at all possible")
print("4. Cross-datacenter round trips are expensive (~150ms)")

## Availability Numbers

In [ ]:
# Availability / SLA table
nines_data = [
    ("99%",     2),
    ("99.9%",   3),
    ("99.99%",  4),
    ("99.999%", 5),
]

print(f"{'Availability':<15} {'Nines':<8} {'Downtime/year':<22} {'Downtime/day':<15}")
print("-" * 62)
for avail_str, nines in nines_data:
    unavail = 1 - float(avail_str.strip("%")) / 100
    minutes_per_year = unavail * 365.25 * 24 * 60
    minutes_per_day = unavail * 24 * 60
    if minutes_per_year >= 60:
        yr_str = f"{minutes_per_year/60:.1f} hours"
    else:
        yr_str = f"{minutes_per_year:.1f} minutes"
    print(f"{avail_str:<15} {nines:<8} {yr_str:<22} {minutes_per_day:.2f} min")

print("\nCloud provider SLAs: AWS, GCP, Azure typically guarantee 99.9%+")

## Estimation Example: Twitter-like Service

In [ ]:
# Twitter-like QPS and Storage Estimation
print("=== Twitter-like Service Estimation ===")
print()

# Assumptions
mau = 300_000_000          # 300M monthly active users
dau_pct = 0.50             # 50% are daily active
tweets_per_user_per_day = 2
media_tweet_pct = 0.10     # 10% of tweets contain media
retention_years = 5

dau = int(mau * dau_pct)
print(f"Monthly Active Users:  {mau:>15,}")
print(f"Daily Active Users:    {dau:>15,}")
print()

# QPS
tweets_per_day = dau * tweets_per_user_per_day
qps = tweets_per_day / (24 * 3600)
peak_qps = qps * 2

print("--- QPS ---")
print(f"Tweets per day:        {tweets_per_day:>15,}")
print(f"Average QPS:           {qps:>15,.0f}")
print(f"Peak QPS (2x):         {peak_qps:>15,.0f}")
print()

# Storage
tweet_id_bytes = 64
text_bytes = 140
media_bytes = 1_000_000  # 1 MB

media_storage_per_day = dau * tweets_per_user_per_day * media_tweet_pct * media_bytes
media_storage_per_day_tb = media_storage_per_day / (1024**4)

total_media_storage_pb = media_storage_per_day_tb * 365 * retention_years / 1024

print("--- Storage ---")
print(f"Media storage/day:     {media_storage_per_day_tb:>15,.1f} TB")
print(f"Media storage (5 yr):  {total_media_storage_pb:>15,.1f} PB")
print()

# Text storage (much smaller)
text_per_day = dau * tweets_per_user_per_day * (tweet_id_bytes + text_bytes)
text_5yr_tb = text_per_day * 365 * retention_years / (1024**4)
print(f"Text storage (5 yr):   {text_5yr_tb:>15,.1f} TB  (negligible vs media)")

## High-Level Estimation Framework

```mermaid
graph TD
    A[Define Assumptions] --> B[Calculate QPS]
    A --> C[Calculate Storage]
    A --> D[Calculate Bandwidth]

    B --> B1[DAU x actions/user/day]
    B1 --> B2[divide by 86400 seconds]
    B2 --> B3[x peak factor 2-5x]

    C --> C1[Per-item size x items/day]
    C1 --> C2[x retention period]

    D --> D1[QPS x avg response size]

    B3 --> E[Verify Design Meets Requirements]
    C2 --> E
    D1 --> E
```

## Tips for Estimation

| Tip | Why |
|-----|-----|
| **Round aggressively** | 99,987 / 9.1 becomes 100,000 / 10 |
| **Write down assumptions** | Reference them later, show your work |
| **Label all units** | "5" is ambiguous; "5 MB" is not |
| **Start with DAU** | Most estimates flow from daily active users |
| **Use powers of 10** | Million = 10^6, Billion = 10^9 for quick math |
| **Sanity check** | Does the result make sense? Compare to known systems |

## Takeaways

1. **Estimation is about the process**, not precision -- show structured thinking
2. **Memorize key latency numbers** to make informed design decisions
3. **Powers of two** are your building blocks for all data calculations
4. **Availability nines** translate directly to allowed downtime -- know the SLA table
5. **Always state assumptions** before calculating

## See Also

- [[power-of-two]]
- [[latency-numbers]]
- [[availability-numbers]]
- [[qps-storage-estimation]]